# 02: Source selection

**Phase 2: which dataset do I build from?**

Checking:
- raw `ts` (circuit-level telemetry) and
- `structured_data` (or `structured_data_v2_flex_included`, site-level, PV-only by construction). 

### Expected Athena scan

| Cell | What it does | Expected scan |
|---|---|---|
| Fresh partition probe (`ts` + 3 `structured_data` variants) | `$partitions` metadata, same free mechanism as notebook 01 | ~4 x 10 MB minimum, negligible |
| `information_schema.columns` for `solar_analytics_iceberg` | one metadata call, reused for two schema checks | ~10 MB minimum |
| `meta_up23c` circuit counts by `is_pv` | full scan of a small (424k-row) dimension table | well under 10 MB, rounds up to the minimum |
| 5 sample circuit rows from `meta_up23c` | same small table | 10 MB minimum |
| **Real load-circuit sample from `ts`** (flagged below) | one month, `is_pv=false`, filtered to 5 specific `circuit_id`s | **usually small — Iceberg's per-file `circuit_id` min/max stats (visible in notebook 01's raw `$partitions` output) let Athena skip most files. Worst case if that pruning doesn't help: ~8.5 GB (one month's full `is_pv=false` partition, all 17 postcode buckets). At Sydney's ~AUD $8/TB that worst case is still only ~7 cents — flagged because you asked to know the BYTES regardless of dollar cost, not because it's expensive.** Set `RUN_SAMPLE_QUERY = False` in that cell to skip it.



## Setup


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from bms_sa_review.ami_data_analysis.config import ami_config as Config
from bms_sa_review.ami_data_analysis.lib import ami_athena as Athena
from bms_sa_review.ami_data_analysis.lib import ami_inventory as Inventory
from bms_sa_review.ami_data_analysis.lib import ami_sources as Sources

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

Athena.reset_scan_log()
Athena.require_credentials()
print("Credentials OK. Starting source selection.")


## 1. Options

- **`ts`** (`solar_analytics_iceberg.ts`): raw circuit-level telemetry, joined to `meta_up23c` for site_id and circuit_polarity. 
Confirmed in Phase 1:  16,345,254,058 rows, ~457 GB compressed, 24 months (2024-01 .. 2025-12), and
  **52.1% of rows are `is_pv=false` (load)**
- **`structured_data_v2_flex_included`** (`Config.TABLES["structured_data"]`) —
  site-level, built by `build_structured_data.py`, which filters
  `ts.is_pv = True` before ever summing to site level.
- **Everything else Notebook 1 found** (`all_uncurtailedpv*`, `conformance_*`, `pv_ghi_norm_model*`, `split_days*`, `lso_*`)

## 2. Is `structured_data` structurally capable of carrying load, at all?

Independent of any row it contains: does its SCHEMA even have an `is_pv` column? 
If not, no query against it could ever recover load data. 
The exclusion happened upstream, when the table was built.

One `information_schema.columns` call covers the whole `solar_analytics_iceberg` database.
Reused below for both `ts` and the structured_data target rather than querying twice.


In [ ]:
sai_schema = Inventory.column_inventory(Config.SAI)

structured_target = Config.TABLES["structured_data"]

ts_check = Sources.verify_is_pv_only(sai_schema[sai_schema.table_name == "ts"])
structured_check = Sources.verify_is_pv_only(
    sai_schema[sai_schema.table_name == structured_target]
)

print(f"ts:                 {ts_check}")
print(f"{structured_target}: {structured_check}")

assert ts_check["is_pv_only"] is False, (
    "ts has no is_pv column?! That contradicts notebook 01's partition finding -- "
    "stop and investigate before trusting anything below."
)


**Read this before continuing.** If `structured_check["is_pv_only"]` above
came back anything other than `True`, the schema has changed since Phase 1 and
the rest of this notebook's conclusion should not be trusted as written — tell
me rather than proceeding.


## 3. Fresh, self-contained size and row counts

This notebook does not assume notebook 01's kernel state survived. Same free
`$partitions` mechanism, scoped to just the four tables this phase compares, and
cross-checked against Phase 1's recorded findings — a mismatch would mean the
catalogue changed since Phase 1 ran.


In [ ]:
catalog = Inventory.glue_inventory()

fresh_totals, fresh_raw, fresh_log = Inventory.probe_partitions(
    catalog,
    only=["ts", "structured_data", "structured_data_v2", "structured_data_v2_flex_included"],
)
display(fresh_log[["table", "n_partitions", "declared_partition_keys",
                    "actual_partition_columns", "error"]])


In [ ]:
ts_key = f"{Config.SAI}.ts"
structured_key = f"{Config.SAI}.{structured_target}"

ts_stats = fresh_totals.get(ts_key, {})
if ts_stats.get("n_rows") == Config.TS_TOTAL_ROWS:
    print(f"ts row count matches Phase 1's recorded finding: {Config.TS_TOTAL_ROWS:,}")
else:
    print(f"MISMATCH: ami_config recorded {Config.TS_TOTAL_ROWS:,} rows for ts, "
          f"a fresh probe now shows {ts_stats.get('n_rows')!r}. The catalogue has "
          "changed since Phase 1 -- update ami_config.py before trusting anything "
          "downstream of this.")

structured_stats = fresh_totals.get(structured_key, {})
print(f"\n{structured_target}: {structured_stats.get('n_rows', 0):,} rows, "
      f"{Athena.fmt_bytes(structured_stats.get('size_bytes'))}")


## 4. Circuit-count share vs row-volume share

Phase 1's 52.1%/47.9% split is by ROW volume across 24 months. Here we get the
split by CIRCUIT COUNT from `meta_up23c` — a different measure of the same
fleet, and it need not agree exactly (a circuit reporting less completely
contributes fewer rows than its circuit-count share implies). `meta_up23c` is a
small dimension table — no partition predicate needed, cheap regardless.


In [ ]:
circuit_counts = Athena.aq(
    """
    SELECT is_pv, count(*) AS n_circuits, count(DISTINCT site_id) AS n_sites
    FROM meta_up23c
    GROUP BY is_pv
    """,
    database=Config.SAI, label="meta_up23c circuit counts by is_pv",
)
display(circuit_counts)

share_comparison = Sources.compare_circuit_and_row_shares(
    circuit_counts,
    ts_rows_false=Config.TS_ROWS_BY_IS_PV["is_pv=false (load)"],
    ts_rows_true=Config.TS_ROWS_BY_IS_PV["is_pv=true (pv)"],
)
display(share_comparison)


## 5. A real sample of load-circuit data

Statistics establish that load rows exist; they don't show what they look
like. This pulls a handful of real `is_pv=false` circuits and a month of their
actual `power`/`voltage`/`energy_reactive` values.

**Cost note — read before running the next cell.** The dimension-table lookup
below is free-equivalent. The `ts` sample after it adds an explicit
`circuit_id IN (...)` filter — Iceberg tracks per-file min/max `circuit_id`
statistics (visible directly in notebook 01's raw `$partitions` output, in the
`data` column), so Athena can usually skip most files entirely rather than
scanning the whole month. Usually cheap; not formally guaranteed. Worst case
(no pruning at all): ~8.5 GB, ~7 cents AUD. Set `RUN_SAMPLE_QUERY = False` below
to skip this cell entirely if you'd rather not risk it.


In [ ]:
sample_circuits = Athena.aq(
    """
    SELECT circuit_id, site_id, circuit_polarity, circuit_type, ac_capacity_kw, s_99
    FROM meta_up23c
    WHERE is_pv = false
    LIMIT 5
    """,
    database=Config.SAI, label="sample is_pv=false circuits",
)
display(sample_circuits)


In [ ]:
RUN_SAMPLE_QUERY = True  # set False to skip -- see the cost note above

if RUN_SAMPLE_QUERY:
    circuit_ids = ", ".join(str(int(c)) for c in sample_circuits.circuit_id)
    ts_sample = Athena.aq(
        f"""
        SELECT circuit_id, t_stamp, power, voltage, energy_reactive
        FROM ts
        WHERE year = 2025 AND month = 6 AND is_pv = false
          AND circuit_id IN ({circuit_ids})
        ORDER BY circuit_id, t_stamp
        LIMIT 50
        """,
        database=Config.SAI, label="ts sample, is_pv=false circuits",
    )
    merged = ts_sample.merge(
        sample_circuits[["circuit_id", "circuit_polarity", "circuit_type"]],
        on="circuit_id", how="left",
    )
    # Illustrative only -- Phase 3 is where the sign convention is decided and
    # written down once. Shown here so you can eyeball plausibility, not as a
    # methodological choice.
    merged["power_corrected_kw_illustrative"] = merged.power * merged.circuit_polarity / 1000
    display(merged)
else:
    print("RUN_SAMPLE_QUERY is False -- skipped.")


## 6. The trade-off table

Granularity, both signals, and cost — assembled from the evidence gathered
above via `Sources.build_comparison_table`, not hand-typed.


In [ ]:
candidates = [
    Sources.SourceCandidate(
        name="raw `ts` + `meta_up23c`",
        grain="circuit, 5-min",
        has_load_signal=not ts_check["is_pv_only"],
        has_pv_signal=True,
        decomposable=True,  # circuit-level -- signals CAN be told apart, given a
                            # correct circuit-to-signal map (Phase 3's job, not yet done)
        n_rows=ts_stats.get("n_rows"),
        size_bytes=ts_stats.get("size_bytes"),
        cleanliness_notes=(
            "raw power (instantaneous W) and energy_reactive (5-min kvarh) resample "
            "differently -- see ami_config.SOURCE_COLUMN_UNITS; circuit_polarity sign "
            "correction required; meta_up23c fans out 2.47x over circuits, GROUP BY "
            "circuit_id + max(...) required before joining (Phase 1 finding)"
        ),
        complexity_notes=(
            "circuit-to-signal mapping and aggregate-circuit (double-counting) "
            "detection are NEW work -- the existing pipeline never needed this, "
            "because it only ever consumed the PV half"
        ),
    ),
    Sources.SourceCandidate(
        name=f"`{structured_target}` (site-level, PV-only)",
        grain="site, 5-min",
        has_load_signal=not structured_check["is_pv_only"],
        has_pv_signal=True,
        decomposable=False,  # even if load were present, one P_kw_norm column per
                             # site cannot be split back into components
        n_rows=structured_stats.get("n_rows"),
        size_bytes=structured_stats.get("size_bytes"),
        cleanliness_notes=(
            "already site-summed, capacity-normalized (p_kw_norm, not kW -- needs "
            "de-normalizing), clear-sky/GHI enriched, voltage-bounded -- far less "
            "post-processing needed IF it had what this project needs"
        ),
        complexity_notes="ruled out on has_load_signal / decomposable -- cost is moot",
    ),
]

comparison = Sources.build_comparison_table(candidates)
display(comparison)

verdict = Sources.recommend(candidates)
print("\nQualifying candidate(s):", verdict["qualifying"])
print("Excluded:")
for name, reason in verdict["excluded"].items():
    print(f"  - {name}: {reason}")


## 7. What this cost


In [ ]:
display(Athena.scan_report())